In [1]:
import tensorflow.keras as keras
import numpy as np

2026-02-26 21:59:59.306277: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-26 21:59:59.934511: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-26 22:00:02.067530: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [41]:
keras.backend.set_image_data_format('channels_last')

def block(last, i):
    x = keras.layers.Conv2D(256, (3, 3), padding="same", name="block{}_conv{}".format(i, 1), activation="relu",data_format='channels_last')(last)
    x = keras.layers.Conv2D(256, (3, 3), padding="same", name="block{}_conv{}".format(i, 2), activation="linear",data_format='channels_last')(x)
    add = keras.layers.Add()([x, last])
    rel = keras.layers.ReLU()(add)
    return rel, add


inputs = keras.Input((8, 8, 112))
last = keras.layers.Conv2D(256, (3, 3), padding="same", name="inputconv",data_format='channels_last')(inputs)
last = keras.layers.ReLU()(last)
inter_outputs = [last]
for b in range(0, 19):
    last, inter = block(last, b)
    inter_outputs.append(inter)

policy_conv = keras.layers.Conv2D(32, (1, 1), activation="relu", padding="same", name="policy_conv")(last)
flatten = keras.layers.Flatten(data_format="channels_last")(policy_conv)
policy_output = keras.layers.Dense(1858, activation="linear", name="policy_dense")(flatten)

value_conv = keras.layers.Conv2D(32, (1, 1), activation="relu", padding="same", name="value_conv")(last)
flatten = keras.layers.Flatten(data_format="channels_last")(value_conv)
value_dense = keras.layers.Dense(128, activation="relu", name="value_dense_1")(flatten)
value_output = keras.layers.Dense(1, activation="tanh", name="value_output")(value_dense)

model = keras.Model(inputs, [policy_output, value_output])

to_RNN = 64; activ_to_rnn = "tanh"
out_layer = keras.layers.Dense(to_RNN, activation=activ_to_rnn, name="to_rnn_out")(last)
test_model = keras.Model(inputs, outputs=out_layer)

In [42]:
import onnx

from onnx import numpy_helper
_model = onnx.load("models/mod3.onnx")
INTIALIZERS = _model.graph.initializer
weights = {}
for initializer in INTIALIZERS:
    W = numpy_helper.to_array(initializer)
    #weights[initializer.name[1:]]=W
    if len(W.shape) > 2:
        weights[initializer.name[1:]] = np.moveaxis(W, [0, 1], [-1, -2])
    else:
        weights[initializer.name[1:]] = W

In [4]:
print(weights.keys())

dict_keys(['inputconv/w/kernel', 'inputconv/w/bias', 'block0/conv1/w/kernel', 'block0/conv1/w/bias', 'block0/conv2/w/kernel', 'block0/conv2/w/bias', 'const/se_reshape', 'block0/conv2/se/matmul1/w', 'block0/conv2/se/add1/w', 'block0/conv2/se/matmul2/w', 'block0/conv2/se/add2/w', 'block1/conv1/w/kernel', 'block1/conv1/w/bias', 'block1/conv2/w/kernel', 'block1/conv2/w/bias', 'block1/conv2/se/matmul1/w', 'block1/conv2/se/add1/w', 'block1/conv2/se/matmul2/w', 'block1/conv2/se/add2/w', 'block2/conv1/w/kernel', 'block2/conv1/w/bias', 'block2/conv2/w/kernel', 'block2/conv2/w/bias', 'block2/conv2/se/matmul1/w', 'block2/conv2/se/add1/w', 'block2/conv2/se/matmul2/w', 'block2/conv2/se/add2/w', 'block3/conv1/w/kernel', 'block3/conv1/w/bias', 'block3/conv2/w/kernel', 'block3/conv2/w/bias', 'block3/conv2/se/matmul1/w', 'block3/conv2/se/add1/w', 'block3/conv2/se/matmul2/w', 'block3/conv2/se/add2/w', 'block4/conv1/w/kernel', 'block4/conv1/w/bias', 'block4/conv2/w/kernel', 'block4/conv2/w/bias', 'block4

In [ ]:
for layer in model.layers:
    if layer.name == "value_dense_1":
        layer.set_weights([weights["value/dense1/matmul/w"], weights["value/dense1/add/w"]])
    elif layer.name == "value_output":
        layer.set_weights([weights["value/dense2/matmul/w"], weights["value/dense2/add/w"]])
    elif "value_conv" in layer.name:
        layer.set_weights([weights["value/conv/w/kernel"], weights["value/conv/w/bias"]])
    elif "policy_conv" in layer.name:
        layer.set_weights([weights["policy/conv/w/kernel"], weights["policy/conv/w/bias"]])
    elif "policy_dense" in layer.name:
        layer.set_weights([weights["policy/dense/matmul/w"], weights["output/policy/w"]])
    elif "conv" in layer.name:
        layer.set_weights([weights[layer.name.replace('_', '/') + "/w/kernel"], weights[layer.name.replace('_', '/') + "/w/bias"]])

In [37]:
print(model.layers[3].input)

<KerasTensor shape=(None, 8, 8, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_350>


In [ ]:
print(model.summary())

In [43]:
for layer in test_model.layers:
    
    if "conv" in layer.name:
        layer.set_weights([weights[layer.name.replace('_', '/') + "/w/kernel"], weights[layer.name.replace('_', '/') + "/w/bias"]])
    

In [47]:
test_model.compile()
import numpy as np
t_inp = np.random.rand(1,8,8,112)
print(model.predict(t_inp))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
[array([[ 19004.877 , -26057.152 ,  12179.313 , ...,  -5486.3853,
        -14113.017 ,   6039.24  ]], shape=(1, 1858), dtype=float32), array([[-1.]], dtype=float32)]


In [48]:
test_model.save('models/tf_model_19x256.keras')